# 07 — Autotuned Triton QLoRA (Rank 8)

Phase 4 of the project. Same NF4-fused Triton kernels as notebook 06, but wrapped in `@triton.autotune` over a small grid of `(BLOCK_M, BLOCK_N, num_warps, num_stages)` configs. Triton benchmarks every config the first time it sees a new `(M, N, K)` shape and caches the winner — the same trick cuBLAS uses internally and the lowest-effort way to close the throughput gap to plain LoRA.

**Scope:** rank 8 only, five methods. We compare against the static-tile Triton kernel from notebook 06 directly. Extend the experiment grid (rank 16, 32) once the rank-8 numbers look right.

**Important:** Triton autotune pays a one-time cost the first time each matmul shape appears. For TinyLlama there are roughly 7 distinct linear shapes (q/k/v/o, gate, up, down). The notebook runs a short *warmup* pass before the timed training so that all shapes are tuned and cached, otherwise the first epoch's tokens/sec is polluted by tuning overhead.

In [ ]:
from pathlib import Path
import sys
import torch

PROJECT_ROOT = Path("/content/qLoRA").resolve()
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
import subprocess

subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "origin", "main"], check=False)


In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "requirements.txt")])


In [ ]:
import subprocess, sys
try:
    import triton  # noqa: F401
    print(f"triton already available: {triton.__version__}")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "triton"])
    import triton
    print(f"installed triton: {triton.__version__}")


## 1. Probe Triton + CUDA

In [ ]:
import torch
from qlora_scratch import TRITON_AVAILABLE

print(f"TRITON_AVAILABLE: {TRITON_AVAILABLE}")
print(f"CUDA available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:              {torch.cuda.get_device_name(0)}")
    print(f"Compute capability: {torch.cuda.get_device_capability(0)}")
assert TRITON_AVAILABLE and torch.cuda.is_available(), (
    "This notebook requires Triton + CUDA. Switch the Colab runtime to a GPU."
)


## 2. Sanity Check: Autotuned Triton vs Dense

Verify the autotuned kernels produce the same outputs and gradients as the dense reference. The first call benchmarks all configs and may take a few seconds — that's the autotune doing its job.

In [ ]:
import torch
from qlora_scratch.lora import (
    LoRAConfig,
    QuantizedLoRALinear,
    AutotunedTritonQuantizedLoRALinear,
)

torch.manual_seed(0)
device = torch.device("cuda")

ref = torch.nn.Linear(512, 768, bias=True).to(device).half()
cfg = LoRAConfig(rank=8, alpha=16, dropout=0.0, block_size=64, chunk_size=128)

dense = QuantizedLoRALinear(ref, cfg).to(device)
auto  = AutotunedTritonQuantizedLoRALinear(ref, cfg).to(device)
auto.load_state_dict(dense.state_dict())

print(f"Using Triton kernel: {auto.using_triton_kernel}")

x  = torch.randn(4, 32, 512, dtype=torch.float16, device=device, requires_grad=True)
x2 = x.detach().clone().requires_grad_(True)

y_dense = dense(x)
y_auto  = auto(x2)
fwd_err = (y_dense - y_auto).abs().max().item()
print(f"forward max |err|: {fwd_err:.3e}")

g = torch.randn_like(y_dense)
y_dense.backward(g); y_auto.backward(g)
gx_err = (x.grad - x2.grad).abs().max().item()
gA_err = (dense.lora_A.grad - auto.lora_A.grad).abs().max().item()
gB_err = (dense.lora_B.grad - auto.lora_B.grad).abs().max().item()
print(f"grad_x max |err|: {gx_err:.3e}")
print(f"grad_A max |err|: {gA_err:.3e}")
print(f"grad_B max |err|: {gB_err:.3e}")
assert fwd_err < 1e-2 and gx_err < 1e-2 and gA_err < 1e-2 and gB_err < 1e-2
print("Autotuned Triton kernel matches the dense reference within fp16 tolerance.")


## 3. Prepare Dataset

## 1. Build OASST1 Splits

In [ ]:
from qlora_scratch.data import build_oasst1_splits

dataset = build_oasst1_splits(DATA_DIR)
dataset


In [ ]:
import json
from pathlib import Path

print(json.loads((DATA_DIR / "metadata.json").read_text()))


## 2. Quick Dataset Inspection

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

train_lengths = [len(text) for text in dataset["train"]["text"]]
stats = pd.Series(train_lengths).describe()
print(stats.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(train_lengths, bins=50, edgecolor="black", alpha=0.7)
axes[0].set_title("Text Length Distribution")
axes[0].set_xlabel("Characters")
axes[0].set_ylabel("Count")

word_counts = [len(text.split()) for text in dataset["train"]["text"]]
axes[1].hist(word_counts, bins=50, edgecolor="black", alpha=0.7, color="orange")
axes[1].set_title("Word Count Distribution")
axes[1].set_xlabel("Words")
axes[1].set_ylabel("Count")
plt.tight_layout()
plt.show()


## 4. Autotune Warmup

Run a few forward + backward passes on the actual TinyLlama shapes so Triton can pick the best config for each linear layer **before** the timed training loop starts. Without this, the first epoch's wall time would include kernel benchmarking and you'd underestimate steady-state throughput.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from qlora_scratch.lora import LoRAConfig, prepare_model_for_autotuned_triton_kbit_training

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading model for warmup...")
warmup_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

cfg = LoRAConfig(rank=8, alpha=16, dropout=0.05, block_size=64, chunk_size=128)
warmup_model = prepare_model_for_autotuned_triton_kbit_training(warmup_model, cfg)
warmup_model.to("cuda")

batch = tokenizer(
    ["Warmup pass for Triton autotune"] * 2,
    return_tensors="pt",
    padding="max_length",
    truncation=True,
    max_length=256,
).to("cuda")
batch["labels"] = batch["input_ids"]

warmup_model.train()
print("Running 3 warmup steps to trigger autotune for all matmul shapes...")
for i in range(3):
    out = warmup_model(**batch)
    out.loss.backward()
    warmup_model.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    print(f"  warmup step {i+1}: loss={out.loss.item():.3f}")

del warmup_model
import gc; gc.collect(); torch.cuda.empty_cache()
print("Autotune cache populated.")


## 5. Build Rank-8 Experiment Grid (5 Methods)

In [ ]:
from qlora_scratch.train import ExperimentConfig

sample_prompts = [
    "### Instruction:\nSummarize QLoRA in plain language for a student.\n\n### Response:\n",
    "### Instruction:\nWhat is the purpose of NF4 quantization in QLoRA?\n\n### Response:\n",
    "### Instruction:\nExplain how Triton autotune picks tile sizes per matmul.\n\n### Response:\n",
]

methods = ["lora", "qlora", "qlora_chunked", "qlora_triton", "qlora_triton_autotuned"]

base_kwargs = dict(
    model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    lora_rank=8,
    lora_alpha=16,
    quant_block_size=64,
    max_train_samples=512,
    max_eval_samples=128,
)

experiments = {}
for method in methods:
    name = f"scratch_{method}_r8"
    kwargs = dict(
        method=method,
        output_dir=str(RESULTS_DIR / "autotuned_run" / name),
        **base_kwargs,
    )
    if method in ("qlora_chunked", "qlora_triton", "qlora_triton_autotuned"):
        kwargs["quant_chunk_size"] = 128
    experiments[name] = ExperimentConfig(**kwargs)

print(f"Total experiments: {len(experiments)}")
for name, cfg in experiments.items():
    print(f"  {name:42s} method={cfg.method}")


## 6. Run Experiments

In [ ]:
import gc, json
from pathlib import Path
import torch
from qlora_scratch.train import run_experiment

all_metrics = []
for exp_name, config in experiments.items():
    print("\n" + "#" * 78)
    print(f"# {exp_name}  (method={config.method})")
    print("#" * 78)
    metrics = run_experiment(PROJECT_ROOT / "data", config, sample_prompts=sample_prompts)
    metrics["experiment"] = exp_name
    Path(config.output_dir).mkdir(parents=True, exist_ok=True)
    (Path(config.output_dir) / "metrics.json").write_text(json.dumps(metrics, indent=2))
    all_metrics.append(metrics)
    print(
        f"  -> eval_loss={metrics['eval_loss']:.4f}  "
        f"peak_vram={metrics['peak_vram_mb']:.1f} MB  "
        f"tokens/s={metrics['tokens_per_second']:.1f}  "
        f"wall={metrics['wall_time_s']:.1f}s"
    )
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\nFinished {len(all_metrics)} experiments.")


## 7. Full Metrics Table

In [ ]:
import pandas as pd
from qlora_scratch.analysis import results_summary_table

summary = results_summary_table(all_metrics)
display_cols = [
    "experiment", "method",
    "eval_loss", "perplexity",
    "peak_vram_mb", "peak_reserved_vram_mb",
    "tokens_per_second", "avg_generation_tokens_per_second",
    "avg_optimizer_step_latency_s", "wall_time_s",
]
pd.set_option("display.float_format", lambda v: f"{v:.3f}")
summary[display_cols].sort_values("method").reset_index(drop=True)


## 8. Plot Style

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

METHOD_ORDER = ["lora", "qlora", "qlora_chunked", "qlora_triton", "qlora_triton_autotuned"]
METHOD_LABELS = {
    "lora":                   "LoRA",
    "qlora":                  "QLoRA (dense dequant)",
    "qlora_chunked":          "QLoRA (chunked)",
    "qlora_triton":           "QLoRA (Triton fused)",
    "qlora_triton_autotuned": "QLoRA (Triton autotuned)",
}
METHOD_COLORS = {
    "lora":                   "#33658A",
    "qlora":                  "#BC4B51",
    "qlora_chunked":          "#5B8E7D",
    "qlora_triton":           "#F4A259",
    "qlora_triton_autotuned": "#7F5AA8",
}

sns.set_theme(style="whitegrid", context="notebook", font_scale=1.05)
plt.rcParams.update({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "axes.titlesize": 13,
    "axes.labelweight": "semibold",
    "legend.frameon": True,
    "legend.framealpha": 0.92,
    "legend.edgecolor": "#cccccc",
    "figure.dpi": 110,
})

def single_rank_bar(summary, metric, ylabel, title, fmt="{:.0f}"):
    fig, ax = plt.subplots(figsize=(11, 5.5))
    sub = summary.set_index("method").reindex(METHOD_ORDER)
    values = sub[metric].values
    colors = [METHOD_COLORS[m] for m in METHOD_ORDER]
    labels = [METHOD_LABELS[m] for m in METHOD_ORDER]
    bars = ax.bar(
        np.arange(len(METHOD_ORDER)), values,
        color=colors, edgecolor="white", linewidth=0.8,
    )
    for bar, val in zip(bars, values):
        if not np.isnan(val):
            ax.annotate(
                fmt.format(val),
                xy=(bar.get_x() + bar.get_width() / 2, val),
                xytext=(0, 3), textcoords="offset points",
                ha="center", va="bottom", fontsize=9,
            )
    ax.set_xticks(np.arange(len(METHOD_ORDER)))
    ax.set_xticklabels(labels, rotation=18, ha="right")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    handles = [plt.Rectangle((0, 0), 1, 1, color=METHOD_COLORS[m]) for m in METHOD_ORDER]
    ax.legend(handles, labels, title="Method", loc="best", fontsize=9)
    ax.margins(y=0.18)
    fig.tight_layout()
    return fig

print("Style configured.")


## 9. Peak VRAM

In [ ]:
single_rank_bar(
    summary, "peak_vram_mb",
    ylabel="Peak allocated VRAM (MB)",
    title="Peak VRAM at LoRA rank 8",
)


## 10. Training Throughput

In [ ]:
single_rank_bar(
    summary, "tokens_per_second",
    ylabel="Tokens / second",
    title="Training Throughput at LoRA rank 8",
)


## 11. Wall-Clock Training Time

In [ ]:
single_rank_bar(
    summary, "wall_time_s",
    ylabel="Wall-clock time (s)",
    title="Total Training Time at LoRA rank 8",
)


## 12. Training Loss Curves (5 methods)

In [ ]:
def smooth(values, window=15):
    arr = np.asarray(values, dtype=float)
    if len(arr) < window:
        return arr
    return np.convolve(arr, np.ones(window) / window, mode="valid")

fig, ax = plt.subplots(figsize=(11.5, 5.5))
for metrics in all_metrics:
    method = metrics["config"]["method"]
    curve  = smooth(metrics["train_loss_curve"], window=15)
    ax.plot(
        np.arange(len(curve)), curve,
        color=METHOD_COLORS[method],
        linewidth=2.0, alpha=0.92,
        label=METHOD_LABELS[method],
    )
ax.set_xlabel("Mini-batch step (rolling mean, window = 15)")
ax.set_ylabel("Train loss")
ax.set_title("Training Loss Curves at LoRA rank 8")
ax.legend(title="Method", loc="upper right", fontsize=9)
fig.tight_layout()
fig


## 13. Deltas vs Static-Tile Triton (Apples-to-Apples)

In [ ]:
import pandas as pd

def normalize_total_tokens(metrics_list):
    rows = []
    for m in metrics_list:
        wall = m["wall_time_s"]
        tps  = m["tokens_per_second"]
        rows.append({
            "method": m["method"],
            "wall_time_s": wall,
            "tokens_per_second": tps,
            "implied_total_tokens": wall * tps,
            "peak_vram_mb": m["peak_vram_mb"],
            "eval_loss": m["eval_loss"],
        })
    return pd.DataFrame(rows).set_index("method").reindex(METHOD_ORDER)

table = normalize_total_tokens(all_metrics)
print("Per-method snapshot:")
print(table.round(2).to_string())

base_method = "qlora_triton"
if base_method in table.index:
    base_tps  = table.loc[base_method, "tokens_per_second"]
    base_vram = table.loc[base_method, "peak_vram_mb"]
    deltas = pd.DataFrame({
        "tps_delta_vs_static_triton_pct":  ((table["tokens_per_second"] - base_tps) / base_tps * 100).round(2),
        "vram_delta_vs_static_triton_pct": ((table["peak_vram_mb"] - base_vram) / base_vram * 100).round(2),
    })
    print(f"\nDeltas vs {base_method!r} (autotune target):")
    print(deltas.to_string())


## 14. Pre/Post Instruction-Tuning Samples (Autotuned Run)

In [ ]:
from qlora_scratch.analysis import build_instruction_tuning_table

auto_metrics = next(m for m in all_metrics if m["method"] == "qlora_triton_autotuned")
instruction_table = build_instruction_tuning_table(auto_metrics)
instruction_table


In [ ]:
for row in instruction_table.to_dict(orient="records"):
    print(f"=== PROMPT {row['prompt_id']} ===")
    print(row["prompt"])
    print("\n--- PRE-FINETUNE ---")
    print(row["pre_response"])
    print("\n--- POST-FINETUNE ---")
    print(row["post_response"])
    print()
